In [67]:
import mne

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/mne/fixes.py:988: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(numba.__version__) < LooseVersion('0.40'):
/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/mne/fixes.py:988: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(numba.__version__) < LooseVersion('0.40'):


In [68]:
def read_data(path):
  raw = mne.io.read_raw_gdf(path,eog=['EOG-left', 'EOG-central', 'EOG-right'],preload=True)
  raw.drop_channels(['EOG-left', 'EOG-central', 'EOG-right'])
  raw.set_eeg_reference()
  events = mne.events_from_annotations(raw)
  epoch = mne.Epochs(raw,events[0],event_id=[7,8,9,10],tmin = -0.1,tmax=0.7,on_missing='warn')
  labels= epoch.events[:,-1]
  features= epoch.get_data()
  return features,labels

In [ ]:
import numpy as np

data = []
label = []

for i in range(1, 10):
    features, labels = read_data(f'/Users/siddharth/Downloads/BCICIV_2a_gdf/A0{i}T.gdf')
    if len(data) == 0:
        data = features
        label = labels
    else:
        data = np.concatenate((data, features), axis=0)
        label = np.concatenate((label, labels), axis=0)


In [70]:
print(data.shape)
print(label.shape)

(2448, 22, 201)
(2448,)


In [71]:
adjacency_matrices = data
labels=label

In [72]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, GATConv, ChebConv, SAGEConv, global_mean_pool
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

In [73]:
# labels=np.transpose(labels,(1,0))
class BrainConnectivityDataset(Dataset):
    def __init__(self, adjacency_matrices, labels, transform=None, pre_transform=None):
        self.adjacency_matrices = adjacency_matrices
        self.labels = labels
        super(BrainConnectivityDataset, self).__init__(None, transform, pre_transform)

    def len(self):
        return len(self.labels)

    def get(self, idx):
        edge_index = self.adjacency_matrix_to_edge_index(self.adjacency_matrices[idx])
        x = torch.ones((self.adjacency_matrices.shape[1], 1), dtype=torch.float)
        y = torch.tensor([self.labels[idx]], dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, y=y)
        return data

    @staticmethod
    def adjacency_matrix_to_edge_index(matrix):
        edge_index = []
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                if matrix[i, j]:  # If there is an edge
                    edge_index.append([i, j])
        return torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
# Split the dataset into training and testing sets (Thos os no)
train_indices, test_indices = train_test_split(range(len(labels)), test_size=0.2, random_state=42)
train_data = adjacency_matrices[train_indices]
test_data = adjacency_matrices[test_indices]
train_labels = labels[train_indices]
test_labels = labels[test_indices]

# Create the datasets
train_dataset = BrainConnectivityDataset(train_data, train_labels)
test_dataset = BrainConnectivityDataset(test_data, test_labels)

In [84]:
adjacency_matrices.shape[1]

22

In [74]:
print(train_dataset)
print(test_dataset)

BrainConnectivityDataset(1958)
BrainConnectivityDataset(490)


In [75]:
# Create DataLoaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Code Below Does not Works

In [52]:
import torch
from torch.nn import Linear
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(44, 4)
        self.conv2 = GCNConv(4, 4)
        self.conv3 = GCNConv(4, 2)
        self.classifier = Linear(2, 4)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = h.tanh()
        h = self.conv2(h, edge_index)
        h = h.tanh()
        h = self.conv3(h, edge_index)
        h = h.tanh()  # Final GNN embedding space.
        
        # Apply a final (linear) classifier.
        out = self.classifier(h)

        return out, h

In [53]:
# Training loop
model = GCN()
i =1
model.train()

GCN(
  (conv1): GCNConv(44, 4)
  (conv2): GCNConv(4, 4)
  (conv3): GCNConv(4, 2)
  (classifier): Linear(in_features=2, out_features=4, bias=True)
)

In [36]:
model = GCN()
criterion = torch.nn.CrossEntropyLoss()  #Initialize the CrossEntropyLoss function.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Initialize the Adam optimizer.

In [54]:
import torch
from torch.nn import Linear
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        
        # Assuming data.x has shape [44, 1], i.e. 1 input feature
        self.conv1 = GCNConv(1, 4)  # First layer: Input feature dimension is 1
        self.conv2 = GCNConv(4, 4)  # Second layer: Hidden layer size of 4
        self.conv3 = GCNConv(4, 2)  # Third layer: Output to 2 features
        
        # Assuming data.y is [2,1], indicating binary classification (2 classes)
        self.classifier = Linear(2, 2)  # Classifier to predict 2 classes

    def forward(self, x, edge_index, batch):
        h = self.conv1(x, edge_index)
        h = h.tanh()
        h = self.conv2(h, edge_index)
        h = h.tanh()
        h = self.conv3(h, edge_index)
        h = h.tanh()  # Final GNN embedding space
        
        # Apply a final (linear) classifier
        out = self.classifier(h)

        return out, h


# Initialize the model
model = GCN()
print(model)

# Initialize loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training function
def train(data):
    optimizer.zero_grad()  # Clear gradients
    out, h = model(data.x, data.edge_index, data.batch)  # Perform a forward pass
    
    # Compute the loss solely based on the training nodes
    loss = criterion(out, data.y.squeeze())  # Ensure data.y shape matches expected input
    
    loss.backward()  # Derive gradients
    optimizer.step()  # Update parameters based on gradients
    
    return loss, h

# Training loop for multiple epochs
for epoch in range(401):
    for data in train_loader:  # Loop over the batches in train_loader
        loss, h = train(data)
    print(f'Epoch: {epoch}, Loss: {loss}')


GCN(
  (conv1): GCNConv(1, 4)
  (conv2): GCNConv(4, 4)
  (conv3): GCNConv(4, 2)
  (classifier): Linear(in_features=2, out_features=2, bias=True)
)


ValueError: Expected input batch_size (44) to match target batch_size (2).

In [39]:
def check_edge_index(data):
    num_nodes = data.x.size(0)  # Number of nodes (44 in your case)
    if data.edge_index.max() >= num_nodes or data.edge_index.min() < 0:
        raise RuntimeError(f'Invalid edge_index. Found an index out of bounds for num_nodes={num_nodes}. '
                           f'Max index in edge_index: {data.edge_index.max()}, '
                           f'Min index in edge_index: {data.edge_index.min()}')

for data in train_loader:
    check_edge_index(data)  # Check edge_index validity before training

    # Your training loop
    loss, h = train(data)

RuntimeError: Invalid edge_index. Found an index out of bounds for num_nodes=44. Max index in edge_index: 222, Min index in edge_index: 0

In [40]:
def filter_invalid_edges(data):
    num_nodes = data.x.size(0)  # Number of nodes (44 in your case)
    
    # Create a mask to filter valid edges
    valid_mask = (data.edge_index[0] < num_nodes) & (data.edge_index[1] < num_nodes)
    
    # Keep only valid edges
    data.edge_index = data.edge_index[:, valid_mask]
    
    return data

# Apply the filter before training
for data in train_loader:
    data = filter_invalid_edges(data)  # Filter invalid edges
    loss, h = train(data)  # Proceed with the training
    print(f'Epoch: {epoch}, Loss: {loss}')

ValueError: Expected input batch_size (44) to match target batch_size (2).

In [88]:
for epoch in range(200):
   total_loss = 0
   for data in train_loader:
       optimizer.zero_grad()
       output = model(data)
       print(output.shape)
       loss = F.nll_loss(output, data.y.view(-1))
       loss.backward()
       optimizer.step()
       total_loss += loss.item()
       print(f'Data {i} Running')
       i = i+1
   print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

RuntimeError: index 44 is out of bounds for dimension 0 with size 44

In [ ]:
for data in train_loader:
 print(data.x.shape,data.edge_index.shape,data.batch.shape,data.y.shape)

In [69]:
import torch
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
import torch.nn.functional as F


class GCN(torch.nn.Module):
    """
    The graph convolutional operator from the "Semi-supervised
    Classification with Graph Convolutional Networks"
    <https://arxiv.org/abs/1609.09207>
    """

    def __init__(self, num_features):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(num_features, 8)
        self.conv2 = GCNConv(8, 16)

        # self.fc = torch.nn.Linear(2 * 16, 1)
        self.fc = torch.nn.Linear(2 * 16, 2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x = torch.cat([gmp(x, batch), gap(x, batch)], dim=1)
        # x = torch.sigmoid(self.fc(x)).squeeze(1)
        x = self.fc(x)
        
        return x

In [50]:
import torch
import torch.optim as optim
from torch_geometric.loader import DataLoader

# Assuming you have a dataset class and instance named `dataset`
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Initialize model, optimizer, and loss function
model = GNNModel(num_node_features=1, hidden_dim=64, num_classes=4)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Number of epochs
num_epochs = 100

# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    total_loss = 0

    for data in train_loader:
        optimizer.zero_grad()  # Clear gradients
        output = model(data)  # Forward pass
        loss = F.nll_loss(output, data.y.view(-1))  # Calculate loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update model parameters
        total_loss += loss.item()

    # Print average loss for the epoch
    print(f'Epoch: {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader)}')

# Optionally: Save the model after training
torch.save(model.state_dict(), 'gnn_model.pth')


RuntimeError: index 88 is out of bounds for dimension 0 with size 88

In [51]:
total_loss = 0
for data in train_loader:
 data1=data

In [57]:
data.y.shape

torch.Size([2, 1])

In [70]:
import torch
import numpy as np


def train(model, criterion, optimizer, data_loader, device, train_num, epochs, logged=False):
    for epoch in range(epochs):
        model.train()
        
        running_loss = 0.0
        num_correct = 0
        batch_size = None
        
        for index, data in enumerate(train_loader):

            # if index == 0:
            #     np.save('x.npy', data.x)
            #     np.save('edge_index.npy', data.edge_index)
            #     np.save('batch.npy', data.batch)
            #     np.save('edge_weight.npy', data.edge_attr)

            data = data.to(device)
            y = torch.from_numpy(np.asarray(data.y)).float()
            y = y.to(device)
            batch_size = y.shape[0] if index == 0 else batch_size

            y_pred = model(data)
            _, pred = torch.max(y_pred, 1)
            # print(pred, y)
            num_correct += (pred == y).sum()

            loss = criterion(y_pred, y.long())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item())

        batch_num = train_num // batch_size
        _loss = running_loss / (batch_num + 1)
        acc = num_correct.item() / train_num * 100
        
        print(f'Epoch {epoch + 1}/{epochs}\tTrain loss: {_loss:.4f}\t'
              f'Train acc: {acc:.2f}%')

    # path = f'checkpoint/{model.__class__.__name__}_{epochs}.pth'
    # torch.save(model.state_dict(), path)
    print('Finish training!')


def test(model, criterion, data_loader, device, test_num, log, logged=False):
    model.eval()
    
    running_loss = 0.0
    num_correct = 0
    batch_size = None
    
    for index, data in enumerate(test_loader):
        data = data.to(device)
        y = torch.from_numpy(np.asarray(data.y)).float()
        y = y.to(device)
        batch_size = y.shape[0] if index == 0 else batch_size

        y_pred = model(data)
        _, pred = torch.max(y_pred, 1)
        num_correct += (pred == y).sum()

        loss = criterion(y_pred, y.long())
        running_loss += float(loss.item())

    batch_num = test_num // batch_size
    _loss = running_loss / (batch_num + 1)
    acc = num_correct.item() / test_num * 100
    print(f'Test loss: {_loss:.4f}\tTest acc: {acc:.2f}%')

    if logged:
        log.append(f'{acc:.2f}\t\n')
        with open('result/gnn_20210111.txt', 'a') as f:
            f.writelines(log)

In [74]:
for epoch in range(10):
    model.train()
    
    running_loss = 0.0
    num_correct = 0
    batch_size = None
    
    for index, data in enumerate(train_loader):
        y = torch.from_numpy(np.asarray(data.y)).float()
        batch_size = y.shape[0] if index == 0 else batch_size

        y_pred = model(data)
        _, pred = torch.max(y_pred, 1)
        print(pred, y)
        num_correct += (pred == y).sum()

        loss = criterion(y_pred, y.long())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item())

    batch_num = train_num // batch_size
    _loss = running_loss / (batch_num + 1)
    acc = num_correct.item() / train_num * 100
    
    print(f'Epoch {epoch + 1}/{epochs}\tTrain loss: {_loss:.4f}\t'
          f'Train acc: {acc:.2f}%')


RuntimeError: index 44 is out of bounds for dimension 0 with size 44

In [58]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

# Load your EEG data and labels
data = np.load('/Users/siddharth/Downloads/data.npy')  # shape (2448, 22, 201)
labels = np.load('/Users/siddharth/Downloads/labels.npy')  # shape (2448,)

# Define adjacency matrix creation
def create_adjacency_matrix(eeg_sample):
    # This function should return a 22x22 adjacency matrix
    # For now, we can assume a fully connected graph, or you can use your custom logic
    num_channels = eeg_sample.shape[0]  # 22 channels
    adjacency_matrix = np.ones((num_channels, num_channels))  # Full connectivity
    return adjacency_matrix

# Update the feature extraction to retain the time-series dimension or meaningful features
def extract_features(eeg_sample):
    # If you want to keep the full time-series, return it directly
    return eeg_sample.T  # Shape (201, 22) becomes (22, 201)

# Define the updated PyTorch Dataset class
class BrainConnectivityDataset(torch.utils.data.Dataset):
    def __init__(self, adjacency_matrices, features, labels):
        self.adjacency_matrices = adjacency_matrices  # Shape (num_samples, 22, 22)
        self.features = features  # Shape (num_samples, 22, time_points) or (22, 1) if extracted features
        self.labels = labels  # Shape (num_samples,)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        adj_matrix = self.adjacency_matrices[idx]
        feature = torch.tensor(self.features[idx], dtype=torch.float)  # Node features (22, feature_dim)
        label = torch.tensor(self.labels[idx], dtype=torch.long)  # Class label

        # Convert adjacency matrix to edge index
        edge_index = []
        for i in range(adj_matrix.shape[0]):
            for j in range(adj_matrix.shape[1]):
                if adj_matrix[i, j]:  # If there's an edge
                    edge_index.append([i, j])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        return Data(x=feature, edge_index=edge_index, y=label)

# Preprocess the dataset
adjacency_matrices = np.array([create_adjacency_matrix(data[i]) for i in range(len(data))])
features = np.array([extract_features(data[i]) for i in range(len(data))])

# Split into train and test sets
train_indices, test_indices = train_test_split(range(len(labels)), test_size=0.2, random_state=42)
train_data = BrainConnectivityDataset(adjacency_matrices[train_indices], features[train_indices], labels[train_indices])
test_data = BrainConnectivityDataset(adjacency_matrices[test_indices], features[test_indices], labels[test_indices])

# Create PyTorch DataLoaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)


In [65]:
class GCN(torch.nn.Module):
    def __init__(self, num_node_features, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels=22, out_channels=16)
        self.conv2 = GCNConv(16, 32)
        self.fc = torch.nn.Linear(32, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        print(f"x shape: {x.shape}, edge_index shape: {edge_index.shape}")
        # GCN layers
        x = F.relu(self.conv1(x, edge_index))  # First GCN layer
        x = F.relu(self.conv2(x, edge_index))  # Second GCN layer
        
        # Global pooling (mean pooling across nodes)
        x = torch.mean(x, dim=0, keepdim=True)  # If using all nodes' aggregated features
        
        # Fully connected layer
        x = self.fc(x)
        return F.log_softmax(x, dim=1)


# Initialize model
model = GCN(num_node_features=1, num_classes=4)  # EEG features per node are 1-dimensional, 4 classes in total
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()


In [66]:
# Training loop
def train():
    model.train()
    for data in train_loader:
        print(data.x.shape)  # Check shape of input features
        print(data.edge_index.shape)  # Check shape of edge index
        optimizer.zero_grad()
        out = model(data)  # This line throws the error
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()

# Testing loop
def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        with torch.no_grad():
            out = model(data)
            pred = out.argmax(dim=1)
            correct += int((pred == data.y).sum())
    return correct / len(loader.dataset)

# Main training loop
for epoch in range(1, 101):
    loss = train()
    train_acc = test(train_loader)
    test_acc = test(test_loader)
    print(f'Epoch: {epoch}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')


torch.Size([6432, 22])
torch.Size([2, 15488])
x shape: torch.Size([6432, 22]), edge_index shape: torch.Size([2, 15488])


ValueError: Expected input batch_size (1) to match target batch_size (32).